In [1]:
import os
import pandas as pd
from ebmdatalab import bq

In [2]:
sql = f"""
WITH
  drugs AS (
  SELECT
    DATE(i.month) AS month, -- Converts TIMESTAMP TO DATE (AS it's always just 1st OF month) 
    regional_team,
    bnf_name,
    chemical,
    practice,
    SUM(items) AS items,
    SUM(quantity) AS quantity,
  FROM
    ebmdatalab.hscic.normalised_prescribing AS i
  INNER JOIN
    ebmdatalab.hscic.practices prac
  ON
    practice = prac.code
    AND setting = 4
  INNER JOIN
    hscic.bnf AS bnf
  ON
    i.bnf_code = bnf.presentation_code
  WHERE
    i.bnf_code LIKE '0601023AZ%' -- Tirzepatide
  GROUP BY
    month,
    bnf_name,
    regional_team,
    chemical,
    practice)
SELECT
  drugs.month AS month,
  regional_team,
  chemical,
  practice,
  bnf_name,
  items,
  quantity,
FROM
  drugs

"""

# Define the CSV path for caching the results.
csv_path = os.path.join('..', 'data', 'tirzepatide_practice.csv.gz')

# Use the cached_read function from the bq library to run the query.
vtm_matched = bq.cached_read(sql, csv_path=csv_path)

Downloading: 100%|█████████████████████████████████████████████████████████████|
